In [1]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.models.train_model.train_model import *

In [2]:
def train_model(df_train, n_mimo, n_est, m_depth, save_model=False, save_model_folder=None):
    df_selected = select_dataset_features(df_train, "Train")

    feature_selector = Feature_selector(df_selected, target="generation")
    Xs_train, ys_train, name_code_df = feature_selector.get_X_and_y(n_mimo=n_mimo)

    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=n_est, max_depth=m_depth, learning_rate=0.1)
    model.fit(Xs_train, ys_train)
    logger.info(f"Model has been trained successfully")

    ys_pred = model.predict(Xs_train)

    y_pred = get_y_inverse_mimo(df_train, feature_selector, name_code_df, n_mimo, ys_pred)
    y_train = get_y_inverse_mimo(df_train, feature_selector, name_code_df, n_mimo, ys_train)

    rmse_error_train = compute_relative_rmse(y_pred, y_train)
    thresh_error_train = compute_threshold_error(y_pred, y_train)
    rmae_error = compute_relative_mae(y_pred, y_train)
    r2_score = compute_r2_score(y_pred, y_train)
    logger.info(f"Train rmse error: {rmse_error_train:0.3f}%")
    logger.info(f"Train threshold error: {thresh_error_train:0.3f}%")
    logger.info(f"Train rmae error: {rmae_error:0.3f}%")
    logger.info(f"R2 score: {r2_score:0.3f}%")

    if save_model and save_model_folder is not None:
        for filename in os.listdir(save_model_folder):
            os.remove(os.path.join(save_model_folder, filename))

        dump(model, f"{save_model_folder}/model.joblib")

In [3]:
csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df = pd.read_csv(csv_semi_processed_path, encoding='utf-8')

add_is_test_column(df, random_state=42)

save_model = True
save_model_folder = os.path.join(project_root, "src", "models", "fitted_models")
write_predictions = True  # TODO
n_mimo = 4

train_test_ds = Data_selector(Data_selector(df).select_peaks(goodness=3))
train_df = train_test_ds.select_train_test(is_test=False)
test_df = train_test_ds.select_train_test(is_test=True)

train_model(train_df, n_mimo, n_est=2000, m_depth=5, save_model=save_model, save_model_folder=save_model_folder)

model = load_model(save_model_folder)

test_model(model, test_df, n_mimo)


2025-09-29 10:28:50 - train_model - INFO - Train model: Some features have been dropped successfully
2025-09-29 10:29:18 - train_model - INFO - Model has been trained successfully
2025-09-29 10:29:20 - train_model - INFO - Train rmse error: 0.626%
2025-09-29 10:29:20 - train_model - INFO - Train threshold error: 12.545%
2025-09-29 10:29:20 - train_model - INFO - Train rmae error: 0.450%
2025-09-29 10:29:20 - train_model - INFO - R2 score: 1.000%
2025-09-29 10:29:20 - train_model - INFO - Test model: Some features have been dropped successfully
2025-09-29 10:29:22 - train_model - INFO - Test rmse error: 2.033%
2025-09-29 10:29:22 - train_model - INFO - Test threshold error: 39.697%
2025-09-29 10:29:22 - train_model - INFO - Test rmae error: 1.210%
2025-09-29 10:29:22 - train_model - INFO - R2 score: 0.995%
